# Metacritic Game Site Web Scrape - Exploration/Learning

In this notebook I explore and experiment to build a web scraper that will reliably search through the game sites provided in the data set to acquire additional information.

## Planning

We'll try it out on one game before scaling it up to multiple and then the whole data set.

**Test Game**  
Baldur's Gate 3  

**URL**  
https://www.metacritic.com/game/baldurs-gate-3/  

**Which info do we want?**  
- Data from PS5 and XBox Series X platforms
- userscore, positive/negative/mixed/total user review counts


In [1]:
import time
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

from core.config import DATA_FORMATTED_PATH


## Trial 1: Extract a user score

In [2]:
# load formatted data set and extract platforms for which userscore is missing

df = pd.read_csv(DATA_FORMATTED_PATH)

df_bg3 = df.loc[df['title'] == "Baldur's Gate 3", :]

platforms = df_bg3['platform'].loc[df_bg3['userscore'].isna()]

In [41]:
# prepare example url and header
game_url = 'https://www.metacritic.com/game/baldurs-gate-3'

type_suffix = '/user-reviews'
platform_suffix = '?platform=playstation-5'

full_url = game_url + type_suffix + platform_suffix

headers = {
    "User-Agent": "Mozilla/5.0"
}


In [82]:
# try simple request
response = requests.get(full_url, headers=headers)
response.raise_for_status()

In [83]:
# try and get the user score from the ps5 page
html = response.text
soup = BeautifulSoup(html, 'html.parser')

# score matches
score_div = soup.find_all('div', class_ = "score-card-left__score-number")
score_num = score_div[0].find('span').get_text(strip = True)
print(score_num)

8.7


- First attempt at requesting from metacritic page was successful.
- user score was succesfully extracted for the PS5 version of BG3, which has been missing in the data so far

**Insights**  
- A way to extract the prefixes for all possible platforms (as defined by the data) is needed
    - Do a pre-scrape, so to speak, from pages that cover all possible platforms and extract the "slugs" for the platforms ???
    - Create a dict (in the config?) with the mappings to be used in the actual scrape

## Trial 2: Extract platform slugs

In [116]:
# request the main game page content
response_game = requests.get(game_url, headers = headers)
print(response_game)

<Response [200]>


In [118]:
# get the section with the platform list
html = response_game.text
soup = BeautifulSoup(html, 'html.parser')

platforms_list = soup.find_all('div', class_ = 'game-platforms__list')

platform_links = platforms_list[0].find_all(class_ = 'product-score-card--platform')

platform_mapping = {}
for link in platform_links:
    # Extract platform category
    pf_element = link.find('span', class_ = 'game-platform-logo__icon')
    pf_name = pf_element.get('title')
    
    # Extract corresponding slug in the url
    href = link.get('href')
    query = urlparse(href).query
    pf_slug = parse_qs(query)['platform'][0]

    # map them in a dict
    platform_mapping[pf_name] = pf_slug

platform_mapping

{'PC': 'pc',
 'PlayStation 5': 'playstation-5',
 'Xbox Series X': 'xbox-series-x'}

extracting slugs for the BG3 example seems to work!  

**GOAL**: Do a preemptive scrape to extract slugs for all platforms in the data and save the dict somewhere central

In [ ]:
# +++ CREATE FULL PLATFORM - SLUG DICTIONARY +++

# Iterate over all potential platforms to extract the platform slugs
pf_mapping = {}

for pf in df['platform'].unique():

    # Extract rows from the data for the current platform
    candidates = df[df['platform'] == pf]

    # Iterate over candidate games until a slug for the platform is found
    for _, row in candidates.iterrows():

        game = row['url']

        print(f'Search for {pf} slug on {game}')

        # request page information
        response_slug = requests.get(game, headers=headers)
        response_slug.raise_for_status()
        lag = response_slug.elapsed.total_seconds() # log the duration for later

        soup_slug = BeautifulSoup(response_slug.text, 'html.parser')

        # Find the list of platform links on the page
        pf_list = soup_slug.find_all('div', class_ = 'game-platforms__list')
        pf_links = pf_list[0].find_all(class_ = 'product-score-card--platform')

        # iterate over platform link sections to find the one with the platform we need and
        # extract the slug
        for link in pf_links:

            # Extract the relevant element within the section
            pf_element = link.find(class_=['game-platform-logo__icon', 'game-platform-logo__text'])

            # get the platform name from the website code
            if 'game-platform-logo__icon' in pf_element.get('class', []):
                pf_name = pf_element.get('title')
            else:
                pf_name = pf_element.get_text(strip=True)

            if pf_name == pf:
                # Extract href
                href = link.get('href')

                # if there is no url (-> no reviews for that platform on that game page),
                # continue with the next game page
                if href is None:
                    break

                # otherwise get the slug and map it to the platform name from the data
                query = urlparse(href).query
                pf_slug = parse_qs(query)['platform'][0]
                pf_mapping[pf] = pf_slug
                break

        time.sleep(5 * lag) # leave a time gap as to not provoke errors

        # if a slug was found, continue with the next platform
        if pf in pf_mapping:
            break

    # if no slug was found for the platform in any candidate game,
    # write a missing value
    if pf not in pf_mapping:
        pf_mapping[pf] = pd.NA

pf_mapping

Search for PlayStation slug on https://www.metacritic.com/game/tekken-3
Search for Dreamcast slug on https://www.metacritic.com/game/tekken-3
Search for Dreamcast slug on https://www.metacritic.com/game/nfl-2k1
Search for PC slug on https://www.metacritic.com/game/mass-effect-2
Search for Xbox 360 slug on https://www.metacritic.com/game/mass-effect-2
Search for PlayStation 3 slug on https://www.metacritic.com/game/mass-effect-2
Search for PlayStation 5 slug on https://www.metacritic.com/game/baldurs-gate-3
Search for Xbox Series X slug on https://www.metacritic.com/game/baldurs-gate-3
Search for GameCube slug on https://www.metacritic.com/game/resident-evil-4-2005
Search for PlayStation 2 slug on https://www.metacritic.com/game/resident-evil-4-2005
Search for Wii slug on https://www.metacritic.com/game/resident-evil-4-2005
Search for iOS (iPhone/iPad) slug on https://www.metacritic.com/game/bioshock
Search for Xbox slug on https://www.metacritic.com/game/half-life-2
Search for Nintendo

{'PlayStation': 'playstation',
 'Dreamcast': 'dreamcast',
 'PC': 'pc',
 'Xbox 360': 'xbox-360',
 'PlayStation 3': 'playstation-3',
 'PlayStation 5': 'playstation-5',
 'Xbox Series X': 'xbox-series-x',
 'GameCube': 'gamecube',
 'PlayStation 2': 'playstation-2',
 'Wii': 'wii',
 'iOS (iPhone/iPad)': 'ios-iphoneipad',
 'Xbox': 'xbox',
 'Nintendo Switch': 'nintendo-switch',
 'Nintendo 64': 'nintendo-64',
 'Xbox One': 'xbox-one',
 'PlayStation 4': 'playstation-4',
 'Game Boy Advance': 'game-boy-advance',
 'Wii U': 'wii-u',
 'PSP': 'psp',
 'DS': 'ds',
 '3DS': '3ds',
 'PlayStation Vita': 'playstation-vita',
 'Meta Quest': 'meta-quest'}

In [19]:
# save platform-slug mappings to disk
import json

from core.config import MAPPINGS_DIR

with open(MAPPINGS_DIR/'platform_slugs.json', 'w') as f:
    json.dump(pf_mapping, f, indent = 4)

## Trial 3: Iteratively scraping User Score

Use platform slug mapping to iteratively extract user score for BG3 for PS5 and XBox, which have been missing from the data

In [123]:
missing_slugs = platforms.map(platform_mapping).reset_index(drop = True)

for platform, slug in zip(platforms, missing_slugs):
    missing_url = game_url + type_suffix + '/?platform=' + slug
    print(missing_url)

    response = requests.get(missing_url, headers=headers)
    response.raise_for_status()

    lag = response.elapsed.total_seconds()
    
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')

    score_div = soup.find_all('div', class_ = "score-card-left__score-number")
    score_str = score_div[0].find('span').get_text(strip = True)

    print(f'{platform}: {score_str}')

    if score_str != 'tbd':
        mask = df_bg3['platform'] == platform
        df_bg3.loc[mask, 'userscore'] = float(score_str)
    
    time.sleep(5*lag)

https://www.metacritic.com/game/baldurs-gate-3/user-reviews/?platform=playstation-5
PlayStation 5: 8.7
https://www.metacritic.com/game/baldurs-gate-3/user-reviews/?platform=xbox-series-x
Xbox Series X: 7.1


## Trial 4: Accessing Game Stats on main page

In [42]:
response_stats = requests.get(game_url, headers = headers)
response_stats.raise_for_status()

In [ ]:
pos = response_stats.text.find('Unforgiving')
print(response_stats.text[pos-2000 :  pos+2000])

oor",{"total":240,"game_rating_id":240,"label":3906},"Playable",{"total":1424,"game_rating_id":977,"label":3908},"Fair",{"total":1246,"game_rating_id":547,"label":3910},"Good",{"total":3912,"game_rating_id":983,"label":3913},51,"Great",{"total":3325,"game_rating_id":969,"label":3915},"Outstanding",{"total":3917,"game_rating_id":1435,"label":3918},129,"Flawless",[3920,3923,3925,3928,3930,3932,3934,3936],{"total":3921,"label":3922,"play_time":1435},255,">= 80 Hours",{"total":3111,"label":3924,"play_time":983},"~40 Hours",{"total":3926,"label":3927,"play_time":969},20,"~60 Hours",{"total":240,"label":3929,"play_time":977},"~12 Hours",{"total":969,"label":3931,"play_time":547},"~20 Hours",{"total":1238,"label":3933,"play_time":45},"~1 Hour",{"total":45,"label":3935,"play_time":1238},"\u003C 1 Hour",{"total":1238,"label":3937,"play_time":247},"~4 Hours",[3939,3942,3945,3948,3950],{"total":3940,"label":3941,"difficulty_id":247},153,"Tough",{"total":3943,"label":3944,"difficulty_id":240},17,"

**Insights**
- By searching the space around `pos`, one can find the link to a backend page containing information on game stats, including the link to the corresponding gamefaqs page.
- Link for BG3: https://backend.metacritic.com/games/metacritic/1200497541/stats/web?componentName=game-stats&componentDisplayName=Game+Stats&componentType=GameStats
- theoretically, it should be possible to extract all game stat info from that page. 

In [22]:
# Find backend URL 
url_start = response_stats.text.find(r"https:\u002F\u002Fbackend.metacritic.com")

url_end = response_stats.text.find('"', url_start)
stats_url_raw = response_stats.text[url_start:url_end]

stats_url = stats_url_raw.replace(r'\u002F', '/')
stats_url

'https://backend.metacritic.com/footers/metacritic?apiKey=1MOZgmNFxvmljaQR1X9KAij9Mo4xAY3u&componentName=footer&componentType=ContentList&edition=us'

In [ ]:
# Probably more reliable marker to look for, as there could be more urls in there containing
# backend.metacritic
marker = "componentName=game-stats"
pos = response_stats.text.find(marker)
url_start = response_stats.text.rfind('https:',0,pos)
url_end = response_stats.text.find('"',pos)

stats_url_raw = response_stats.text[url_start:url_end]
stats_url = stats_url_raw.replace(r'\u002F', '/')
stats_url

'https://backend.metacritic.com/games/metacritic/1200497541/stats/web?componentName=game-stats&componentDisplayName=Game+Stats&componentType=GameStats'

In [ ]:
# now request the backend page and extract stat data
resp_backend = requests.get(stats_url, headers = headers)
resp_backend.raise_for_status()
stats_json = resp_backend.json() # basically now a python dictionary

diff = stats_json['data']['item']['difficulty']
play_time = stats_json['data']['item']['play_time']
play_status = stats_json['data']['item']['play_status']

In [ ]:
# +++ Now try to do it for a small batch of games +++

# Choose batch of 20 games from the data
unique_games = df[['title', 'url']].drop_duplicates().dropna(subset = ['url'])
sample_games = unique_games.sample(n = 20, random_state = 42)

# marker for game stat backend url in the page code
stat_marker = "componentName=game-stats"

game_stats = {title: {} for title in sample_games['title']}

# iterate over games/urls
for game, url in zip(sample_games['title'], sample_games['url']):
    
    print(f'Working on {game} ({url}) ...')

    # request page info
    resp_stat_trial = requests.get(url, headers = headers)
    resp_stat_trial.raise_for_status()

    # find backend stats url
    pos = resp_stat_trial.text.find(stat_marker)

    # catch if no fit for the backend url is detected
    if pos == -1:
        print(f'  No fitting backend stat URL found for "{game}"')
        game_stats[game] = None
        continue

    stat_url_start = resp_stat_trial.text.rfind('https:',0,pos)
    stat_url_end = resp_stat_trial.text.find('"',pos)

    stats_url_raw = resp_stat_trial.text[stat_url_start:stat_url_end]
    stats_url = stats_url_raw.replace(r'\u002F', '/')

    print(f'  Found backend url for stats: {stats_url}')

    # Request backend page and extract stat data
    resp_backend_trial = requests.get(stats_url, headers = headers)
    resp_backend_trial.raise_for_status()
    stats_json = resp_backend_trial.json() # basically now a python dictionary

    diff = stats_json['data']['item']['difficulty']
    play_t = stats_json['data']['item']['play_time']
    play_stat = stats_json['data']['item']['play_status']

    # gather stats
    print('  gathering stats ...')
    game_stats[game] = {
        'difficulty': diff,
        'play_time': play_t,
        'play_status': play_stat
    }

In [47]:
# Check how many scrapes were successful?
sum(val is None for val in game_stats.values())

0